Z-scores

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==========================================
# CONFIGURATION
# ==========================================
VALIDATION_DIR = r'file_path' 
METADATA_FILE = r'file_path'
MODEL_PATH = "file_path"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 1. THE MODEL (Ensuring Sigma Floor matches Training)
# ==========================================
class QuasarProbabilisticGRU(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.gru = nn.GRU(input_size=2, hidden_size=hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, 2)

    def forward(self, x, dt):
        rnn_input = torch.cat((x, dt), dim=-1)
        out, _ = self.gru(rnn_input)
        params = self.head(out)
        mu_pred = params[:, :, 0:1]
        # MATCH TRAINING: Floor of 0.02
        sigma_pred = F.softplus(params[:, :, 1:2]) + 0.02
        return mu_pred, sigma_pred

# ==========================================
# 2. THE DATASET (Updated with Per-Object Mean Subtraction)
# ==========================================
class QuasarInferenceDataset(Dataset):
    def __init__(self, folder_path, metadata_file):
        self.folder_path = folder_path
        try:
            self.metadata = pd.read_csv(metadata_file)
            self.valid_files = []
            for idx, row in self.metadata.iterrows():
                fname = f"{int(row['dbID'])}.csv"
                if os.path.exists(os.path.join(folder_path, fname)):
                    self.valid_files.append(row)
            
            if len(self.valid_files) == 0:
                self.files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
                self.use_metadata = False
            else:
                self.metadata = pd.DataFrame(self.valid_files)
                self.use_metadata = True
        except:
            self.files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
            self.use_metadata = False

    def __len__(self):
        return len(self.metadata) if self.use_metadata else len(self.files)

    def __getitem__(self, idx):
        if self.use_metadata:
            filename = f"{int(self.metadata.iloc[idx]['dbID'])}.csv"
        else:
            filename = self.files[idx]
        
        file_path = os.path.join(self.folder_path, filename)
        try:
            df = pd.read_csv(file_path)
            col_mag = 'mag_r_sim' if 'mag_r_sim' in df.columns else 'mag'
            mjd = torch.tensor(df['MJD'].values).float()
            mag_raw = torch.tensor(df[col_mag].values).float()

            # --- MEAN STANDARDIZATION ---
            mag_mean = mag_raw.mean()
            mag_centered = mag_raw - mag_mean 
            
            # --- DT CALCULATION & CLAMPING (Match Training) ---
            dt = torch.cat([torch.tensor([0.0]), mjd[1:] - mjd[:-1]])
            dt = torch.clamp(dt, min=1e-3)

            return {
                'mag': mag_centered.unsqueeze(-1), 
                'dt': dt.unsqueeze(-1),
                'dbID': filename
            }
        except:
            return None

def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    return batch[0] if len(batch) > 0 else None

# ==========================================
# MAIN: INFERENCE WITH RECALIBRATION
# ==========================================
def run_recalibrated_inference():
    print("="*40)
    print("STEP 1: INFERENCE & GLOBAL RECALIBRATION")
    print("="*40)
    
    model = QuasarProbabilisticGRU().to(DEVICE)
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        model.eval()
    else:
        print("Model not found.")
        return

    dataset = QuasarInferenceDataset(VALIDATION_DIR, METADATA_FILE)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
    
    print("Pass 1: Collecting all residuals for global stats...")
    all_raw_z_scores = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader):
            if batch is None: continue
            mag = batch['mag'].to(DEVICE).unsqueeze(0)
            dt = batch['dt'].to(DEVICE).unsqueeze(0)
            if mag.size(1) < 2: continue
            
            # Causal Shift: Predict [1:T] using [0:T-1]
            mu_pred, sigma_pred = model(mag[:, :-1, :], dt[:, :-1, :])
            target_mag = mag[:, 1:, :]
            
            # Standardized Residuals
            raw_z = ((target_mag - mu_pred) / sigma_pred).cpu().numpy().flatten()
            all_raw_z_scores.extend(raw_z[np.isfinite(raw_z)])

    all_raw_z_scores = np.array(all_raw_z_scores)
    global_mean = np.mean(all_raw_z_scores)
    global_std = np.std(all_raw_z_scores)
    
    print("-" * 30)
    print(f"CALIBRATION STATS (Global)")
    print("-" * 30)
    print(f"Global Mean: {global_mean:.6f} (Ideal: 0.0)")
    print(f"Global Std:  {global_std:.6f} (Ideal: 1.0)")
    
    print("\nPass 2: Applying calibration to get per-curve Maxima...")
    calibrated_peaks = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Recalibrating Peaks"):
            if batch is None: continue
            mag = batch['mag'].to(DEVICE).unsqueeze(0)
            dt = batch['dt'].to(DEVICE).unsqueeze(0)
            if mag.size(1) < 2: continue
            
            mu_pred, sigma_pred = model(mag[:, :-1, :], dt[:, :-1, :])
            target_mag = mag[:, 1:, :]
            
            # 1. Raw Z
            raw_z = (target_mag - mu_pred) / sigma_pred
            # 2. Calibrated Z (Recenter and Rescale)
            cal_z = (raw_z - global_mean) / global_std
            
            # 3. Save Peak Absolute Deviation
            calibrated_peaks.append(torch.max(torch.abs(cal_z)).item())

    results = np.array(calibrated_peaks)
    np.savez("inference_results_calibrated.npz", 
             max_peaks=results, 
             calibration_mean=global_mean, 
             calibration_std=global_std)
    
    print(f"\nSaved {len(results)} calibrated peaks.")
    
    # 6. Plotting
    plt.figure(figsize=(10, 6))
    plt.hist(results, bins=100, color='teal', alpha=0.7, log=True)
    plt.title(f"Calibrated Z-Scores (Standardized & Floored)")
    plt.xlabel("Sigma (Recalibrated)")
    plt.ylabel("Count (Log Scale)")
    plt.grid(True, alpha=0.3)
    plt.savefig("calibrated_dist.png")
    plt.show()

if __name__ == "__main__":
    run_recalibrated_inference()

QQ Plots & Statistics

In [ ]:
import torch
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import scipy.stats as stats

# ==========================================
# 1. SETUP & MODEL LOADING
# ==========================================
MODEL_PATH = "file_path"
VALIDATION_DIR = r'file_path'
METADATA_FILE = r'file_path'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class QuasarProbabilisticGRU(torch.nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.gru = torch.nn.GRU(input_size=2, hidden_size=hidden_dim, batch_first=True)
        self.head = torch.nn.Linear(hidden_dim, 2)
    def forward(self, x, dt):
        rnn_input = torch.cat((x, dt), dim=-1)
        out, _ = self.gru(rnn_input)
        params = self.head(out)
        mu_pred = params[:, :, 0:1]
        sigma_pred = torch.nn.functional.softplus(params[:, :, 1:2]) + 0.02
        return mu_pred, sigma_pred

model = QuasarProbabilisticGRU().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# ==========================================
# 2. COLLECT ALL STANDARDIZED RESIDUALS (Z)
# ==========================================
print("Collecting all standardized residuals from validation set...")
meta_df = pd.read_csv(METADATA_FILE)
all_z = []

with torch.no_grad():
    for _, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
        file_path = os.path.join(VALIDATION_DIR, f"{int(row['dbID'])}.csv")
        if not os.path.exists(file_path): continue
        
        df = pd.read_csv(file_path)
        mag = torch.tensor(df['mag_r_sim'].values).float().to(DEVICE)
        mag = mag - mag.mean() # Match v3 training centering
        
        mjd = df['MJD'].values
        dt = torch.cat([torch.tensor([0.0]), torch.tensor(mjd[1:] - mjd[:-1])]).float().to(DEVICE)
        dt = torch.clamp(dt, min=1e-3).unsqueeze(-1).unsqueeze(0)
        mag = mag.unsqueeze(-1).unsqueeze(0)

        # --- ADD THIS CHECK ---
        if len(df) < 2:
            print(f"Skipping {row['dbID']}: Sequence too short (length {len(df)})")
            continue

        mu_p, sigma_p = model(mag[:, :-1, :], dt[:, :-1, :])
        z = (mag[:, 1:, :] - mu_p) / sigma_p
        all_z.extend(z.cpu().numpy().flatten())

all_z = np.array(all_z)
# Remove any non-finite values if they exist
all_z = all_z[np.isfinite(all_z)]

# ==========================================
# 3. STATISTICAL ANALYSIS
# ==========================================
mean_z = np.mean(all_z)
var_z = np.var(all_z)
skew_z = stats.skew(all_z)
kurt_z = stats.kurtosis(all_z) # Excess kurtosis (Fisher’s definition, 0 = Gaussian)

print("\n" + "="*40)
print("STATISTICAL MOMENTS OF RESIDUALS (Z)")
print("="*40)
print(f"Mean (μ):     {mean_z:.6f}  (Ideal: 0.0)")
print(f"Variance (σ²): {var_z:.6f}  (Ideal: 1.0)")
print(f"Skewness:      {skew_z:.6f}  (Ideal: 0.0)")
print(f"Kurtosis:      {kurt_z:.6f}  (Ideal: 0.0)")
print("="*40)

# ==========================================
# 4. QQ PLOT VS GAUSSIAN
# ==========================================
print("\nGenerating QQ Plot...")
plt.figure(figsize=(8, 8))
stats.probplot(all_z, dist="norm", plot=plt)
plt.title("Normal Q-Q Plot of Standardized Residuals")
plt.xlabel("Theoretical Quantiles (Gaussian)")
plt.ylabel("Ordered Values (Model Residuals)")
plt.grid(True, alpha=0.3)

# Save the diagnostic plot
plt.savefig("residual_diagnostic_qq.png")
print("Plot saved: residual_diagnostic_qq.png")
plt.show()